## Model-Based RL on LunarLander-v2 with Random-Shooting MPC

This notebook demonstrates:

1. **Data collection**: gather random rollouts from LunarLander-v2  
2. **Dynamics model**: train an MLP to predict Δs = s′ – s  
3. **Surrogate reward**: a simple function of state and action  
4. **MPC controller**: random‐shooting over K sequences of horizon H  
5. **Evaluation**: run the learned‐model MPC in the real env  

## Lunar Lander Environment

The Lunar Lander environment is a classic reinforcement learning problem where the goal is to land a spacecraft on the lunar surface. The environment is simulated using the Box2D physics engine.

**How it Works**

The environment is defined by the following:

* **State Space:** The state of the lander is represented by an 8-dimensional vector that includes the lander's position (x, y), velocity (vx, vy), angle, angular velocity, and whether the left and right legs are in contact with the ground.
* **Action Space:** The lander has 4 discrete actions: do nothing, fire left engine, fire main engine, and fire right engine.
* **Reward Function:** The reward function is designed to encourage the lander to land safely. The lander receives a positive reward for landing on the landing pad and a negative reward for crashing.

  - **Good Return:** ~100-140, Max 200

* **Episode Termination:** An episode ends when the lander lands or crashes.

In [ ]:
!pip install swig
!pip install "gymnasium[box2d]"
!pip install torch matplotlib numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 41.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 16.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for box2d-py: filename=box2d_py-2.3.5-cp311-cp311-linux_x86_64.whl size=2379371 sha256=07000e43e310dbc691403ede5aec0353ee52862ee1835f804b136d19c20e388e
  Stored in directory: /root/.cache/pip/wheels/ab/f1/0c/d56f4a2bdd12bae0a0693ec33f2f0daadb5eb9753c78fa5308
Successfully built box2d-py
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import trange
import matplotlib.pyplot as plt

# create env with rgb_array render mode for Colab animations
ENV_NAME = "LunarLander-v3"
env = gym.make(ENV_NAME, render_mode="rgb_array")
state_dim = env.observation_space.shape[0]   # 8
action_dim = env.action_space.n              # 4 (discrete)

In [ ]:
# --- Visualize 3 full random episodes as animations ---
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# Make sure your env has rgb_array render mode
# env = gym.make(ENV_NAME, render_mode="rgb_array")

for ep in range(3):
    frames = []
    s, _ = env.reset()
    done = False
    # Rollout with random actions, recording each frame
    while not done:
        frames.append(env.render())                # RGB array
        a = env.action_space.sample()
        s2, _, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        s = s2
    # Build animation
    fig = plt.figure(figsize=(4,4))
    plt.axis('off')
    ims = [[plt.imshow(frame, animated=True)] for frame in frames]
    ani = animation.ArtistAnimation(fig, ims, interval=50, blit=True)
    plt.close(fig)
    # Display with a title
    display(HTML(f"<h4>Random Episode {ep+1}</h4>"))
    display(HTML(ani.to_jshtml()))


## 3. Data collection (random policy)

In [ ]:
def collect_random_data(env, num_transitions):
    states, actions, next_states = [], [], []
    s, _ = env.reset()
    for _ in range(num_transitions):
        a = env.action_space.sample()
        s2, _, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        states.append(s)
        actions.append(a)
        next_states.append(s2)
        if done:
            s, _ = env.reset()
        else:
            s = s2
    return np.array(states), np.array(actions), np.array(next_states)

N_INITIAL = 50000
states, actions, next_states = collect_random_data(env, N_INITIAL)
deltas = next_states - states
print("Collected", states.shape[0], "transitions.")


Collected 500000 transitions.


## 4. Dataset and normalization

In [ ]:
# One-hot encode actions
A_oh = np.eye(action_dim)[actions]  # shape (N,4)
X = np.concatenate([states, A_oh], axis=1)
Y = deltas

# Normalize targets
y_mean = Y.mean(0)
y_std  = Y.std(0) + 1e-6
Y_norm = (Y - y_mean) / y_std

# PyTorch DataLoader
dataset = TensorDataset(
    torch.from_numpy(X).float(),
    torch.from_numpy(Y_norm).float()
)
loader = DataLoader(dataset, batch_size=256, shuffle=True)


## 5. Dynamics model

In [ ]:
class DynamicsModel(nn.Module):
    def __init__(self, inp_dim, out_dim, hidden_sizes=[256,512,256]):
        super().__init__()
        layers = []
        last = inp_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

model = DynamicsModel(state_dim+action_dim, state_dim).cuda()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


## 6. Train the model

In [ ]:
EPOCHS = 150
for epoch in range(EPOCHS):
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.cuda(), yb.cuda()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f"Epoch {epoch+1}/{EPOCHS}  avg loss: {total_loss/len(dataset):.5f}")


Epoch 1/150  avg loss: 0.49364
Epoch 2/150  avg loss: 0.44988
Epoch 3/150  avg loss: 0.43292
Epoch 4/150  avg loss: 0.42047
Epoch 5/150  avg loss: 0.41211
Epoch 6/150  avg loss: 0.40499
Epoch 7/150  avg loss: 0.39784
Epoch 8/150  avg loss: 0.39170
Epoch 9/150  avg loss: 0.38592
Epoch 10/150  avg loss: 0.38125
Epoch 11/150  avg loss: 0.37658
Epoch 12/150  avg loss: 0.37300
Epoch 13/150  avg loss: 0.36822
Epoch 14/150  avg loss: 0.36442
Epoch 15/150  avg loss: 0.36152
Epoch 16/150  avg loss: 0.35759
Epoch 17/150  avg loss: 0.35424
Epoch 18/150  avg loss: 0.35138
Epoch 19/150  avg loss: 0.34798
Epoch 20/150  avg loss: 0.34569
Epoch 21/150  avg loss: 0.34291
Epoch 22/150  avg loss: 0.33912
Epoch 23/150  avg loss: 0.33701
Epoch 24/150  avg loss: 0.33474
Epoch 25/150  avg loss: 0.33074
Epoch 26/150  avg loss: 0.32763
Epoch 27/150  avg loss: 0.32563
Epoch 28/150  avg loss: 0.32304
Epoch 29/150  avg loss: 0.32035
Epoch 30/150  avg loss: 0.31752
Epoch 31/150  avg loss: 0.31550
Epoch 32/150  avg

## 7. Surrogate reward

In [ ]:
def surrogate_reward(s, a):
    x,y,vx,vy,ang,vang,leg_l,leg_r = s
    r = - (x**2 + y**2) - 0.1*(vx**2 + vy**2)
    if a != 0:  # any engine fire costs a bit
        r -= 0.3
    return r

## 8. MPC via random shooting

In [ ]:
def mpc_action(state, model, K=500, H=15):
    s_tensor = torch.from_numpy(state).float().cuda()
    best_val, best_a0 = -1e9, 0
    # pre-sample sequences
    seqs = np.random.randint(0, action_dim, size=(K,H))
    for k in range(K):
        s = s_tensor.clone()
        val = 0.0
        for t in range(H):
            a = seqs[k,t]
            val += surrogate_reward(s.cpu().detach().numpy(), a)
            # build input
            a_oh = torch.zeros(action_dim, device='cuda')
            a_oh[a] = 1.0
            inp = torch.cat([s, a_oh]).unsqueeze(0)
            # predict normalized delta
            delta_n = model(inp).squeeze(0)
            # unnormalize
            delta = delta_n * torch.from_numpy(y_std).float().cuda() \
                    + torch.from_numpy(y_mean).float().cuda()
            s = s + delta
        if val > best_val:
            best_val, best_a0 = val, seqs[k,0]
    return int(best_a0)


## 9. Evaluate learned-model MPC

In [ ]:
def run_episode(env, policy_fn, render=False):
    s, _ = env.reset()
    total, done = 0.0, False
    while not done:
        a = policy_fn(s)
        s2, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        total += r
        s = s2
        if render:
            env.render()
    return total

N_EVAL = 5
returns = []
for _ in trange(N_EVAL):
    R = run_episode(env, lambda s: mpc_action(s, model, K=300, H=10))
    returns.append(R)
print("Mean return:", np.mean(returns), "Std:", np.std(returns))


100%|██████████| 5/5 [03:51<00:00, 46.31s/it]

Mean return: -87.99898394019138 Std: 73.30671633720536


## 10. Plot one rollout (animation)

In [ ]:
frames = []
s, _ = env.reset()
done = False
while not done:
    frames.append(env.render())
    a = mpc_action(s, model, K=300, H=10)
    s2, _, terminated, truncated, _ = env.step(a)
    done = terminated or truncated
    s = s2

import matplotlib.animation as animation
from IPython.display import HTML

fig = plt.figure(figsize=(5,4))
ims = [[plt.imshow(f, animated=True)] for f in frames]
ani = animation.ArtistAnimation(fig, ims, interval=50, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())


# Homework Questions
**1. Model Size:**

* How does the size of the dynamics model (number of layers, neurons per layer) influence the performance of the MPC controller? Experiment with different model architectures and discuss the trade-offs between accuracy, training time, and computational cost.

**2. Horizon (H) and Number of Sequences (K):**

* How do the horizon (H) and number of sequences (K) in the random-shooting MPC algorithm affect the controller's performance and computational cost? What are the limitations of using a very long horizon or a very large number of sequences?

**3. Real-Time Control:**

* Discuss the feasibility of using this random-shooting MPC approach for real-time control of the Lunar Lander. What are the potential challenges and how could they be addressed?

**4. Ensemble Models:**

* How could training multiple ensembles of dynamics models improve the robustness and performance of the MPC controller? Discuss different ways to combine predictions from multiple models.


**5. Stochastic Behavior:**

* What are the potential benefits and drawbacks of adding stochastic behavior to the dynamics model or the MPC controller? How could stochasticity be incorporated into the framework?